### Lab 2.2: Perceptron Algorithm in PyTorch

In this lab you will again implement the perceptron algorithm, but this time using PyTorch.

In [1]:
import numpy as np
import torch

PyTorch is very similar to NumPy in its basic functionality.  In PyTorch arrays are called tensors.

In [2]:
a = torch.tensor(5)
a

tensor(5)

In [3]:
b = torch.tensor(6)
a+b

tensor(11)

In [4]:
c = torch.zeros(3,5).float()
c

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

*A note on broadcasting:* You may have noticed in the previous lab that NumPy is particular about the sizes of the arrays in operations; PyTorch is the same way.

For example, if `A` has shape `(10,5)` and `b` has shape `(10,)`, then we can't compute `A*b`.  It wants the *last* dimensions to match, not the first ones.  So you would need to do either `A.T*b`.

In [5]:
A = np.random.normal(size=(10,5))
b = np.ones(10)

In [6]:
try:
    A*b
except ValueError as e:
    print(e)

operands could not be broadcast together with shapes (10,5) (10,) 


In [7]:
A.T*b

array([[ 0.93017598, -0.35079046,  1.20493167, -0.1122061 ,  0.16257488,
        -0.16381387, -0.18797617, -0.03805818, -0.42254129,  0.37529756],
       [-0.25489483,  1.56074058, -0.47995258, -0.55660654,  0.25560236,
        -0.40340176,  0.2945436 ,  0.74659465, -1.60233546,  1.56847418],
       [ 2.36087593,  0.73951439,  0.7671135 , -1.53118203,  0.36067379,
         0.97497091, -0.15004501,  0.94780578, -1.48247972, -0.16266548],
       [-0.058924  , -0.04547907, -0.87190201, -0.24536258,  0.42296043,
         0.26847711, -0.66367388, -1.47259717, -1.36376854, -0.18881499],
       [ 0.89765646, -0.36644411,  0.21921269,  2.18059545, -0.44724009,
         0.60503577,  1.35501789,  0.24761181, -2.10811353, -1.64591622]])

An alternative is to introduce an extra dimension of size one to $b$.  However, note that this produces the transposed result from before.

In [8]:
A*b[:,None]

array([[ 0.93017598, -0.25489483,  2.36087593, -0.058924  ,  0.89765646],
       [-0.35079046,  1.56074058,  0.73951439, -0.04547907, -0.36644411],
       [ 1.20493167, -0.47995258,  0.7671135 , -0.87190201,  0.21921269],
       [-0.1122061 , -0.55660654, -1.53118203, -0.24536258,  2.18059545],
       [ 0.16257488,  0.25560236,  0.36067379,  0.42296043, -0.44724009],
       [-0.16381387, -0.40340176,  0.97497091,  0.26847711,  0.60503577],
       [-0.18797617,  0.2945436 , -0.15004501, -0.66367388,  1.35501789],
       [-0.03805818,  0.74659465,  0.94780578, -1.47259717,  0.24761181],
       [-0.42254129, -1.60233546, -1.48247972, -1.36376854, -2.10811353],
       [ 0.37529756,  1.56847418, -0.16266548, -0.18881499, -1.64591622]])

In [9]:
A*np.expand_dims(b,-1)

array([[ 0.93017598, -0.25489483,  2.36087593, -0.058924  ,  0.89765646],
       [-0.35079046,  1.56074058,  0.73951439, -0.04547907, -0.36644411],
       [ 1.20493167, -0.47995258,  0.7671135 , -0.87190201,  0.21921269],
       [-0.1122061 , -0.55660654, -1.53118203, -0.24536258,  2.18059545],
       [ 0.16257488,  0.25560236,  0.36067379,  0.42296043, -0.44724009],
       [-0.16381387, -0.40340176,  0.97497091,  0.26847711,  0.60503577],
       [-0.18797617,  0.2945436 , -0.15004501, -0.66367388,  1.35501789],
       [-0.03805818,  0.74659465,  0.94780578, -1.47259717,  0.24761181],
       [-0.42254129, -1.60233546, -1.48247972, -1.36376854, -2.10811353],
       [ 0.37529756,  1.56847418, -0.16266548, -0.18881499, -1.64591622]])

In general, carefully check the sizes of all arrays in your code!

In [10]:
from palmerpenguins import load_penguins
from mlxtend.plotting import plot_decision_regions
from matplotlib import pyplot as plt

Here we loading and format the Palmer penguins dataset for binary classification.

In [11]:
df = load_penguins()

# drop rows with missing values
df.dropna(inplace=True)

# tricky code to randomly shuffle the rows
df = df.sample(frac=1).reset_index(drop=True)

# select only two specices
df = df[(df['species']=='Adelie')|(df['species']=='Chinstrap')]

# get two features
X = df[['flipper_length_mm','bill_length_mm']].values

# convert speces labels to 0 and 1
y = df['species'].map({'Adelie':0,'Chinstrap':1}).values

To make the learning algorithm work more smoothly, we we will subtract the mean of each feature.

Here `np.mean` calculates a mean, and `axis=0` tells NumPy to calculate the mean over the rows (calculate the mean of each column).

In [12]:
X -= np.mean(X,axis=0)

Now we will convert our `X` and `y` arrays to torch Tensors.

In [13]:
X = torch.tensor(X).float()
y = torch.tensor(y).float()

In [14]:
X

tensor([[ 6.0794e+00,  9.2953e+00],
        [ 3.0794e+00,  3.9953e+00],
        [ 7.9439e-02, -9.0467e-01],
        [ 7.0794e+00, -1.4047e+00],
        [-6.9206e+00, -5.0047e+00],
        [-3.9206e+00, -9.9047e+00],
        [-1.9206e+00, -3.9047e+00],
        [ 4.0794e+00,  7.9953e+00],
        [-2.9206e+00, -9.0467e-01],
        [-1.1921e+01, -4.3047e+00],
        [-7.9206e+00, -5.4047e+00],
        [-9.2056e-01,  4.3953e+00],
        [-6.9206e+00, -5.0047e+00],
        [ 5.0794e+00,  9.9953e+00],
        [ 9.0794e+00, -5.0467e-01],
        [ 1.3079e+01, -9.0467e-01],
        [-2.9206e+00, -5.1047e+00],
        [-1.1921e+01,  1.9533e-01],
        [ 1.4079e+01,  9.8953e+00],
        [ 3.0794e+00,  3.6953e+00],
        [-1.9206e+00, -6.1047e+00],
        [ 8.0794e+00,  8.4953e+00],
        [-4.9206e+00, -7.5047e+00],
        [-3.9206e+00, -2.5047e+00],
        [-1.0921e+01, -4.4047e+00],
        [-9.2056e-01, -3.0047e+00],
        [ 1.5079e+01,  1.3795e+01],
        [ 4.0794e+00, -2.404

### Exercises

Your task is to again complete this class for the perceptron, with two changes from last time:
- the implementation should use PyTorch tensors, not NumPy arrays;
- `train_step` now accepts the entire dataset as input and should calculate the average gradient over all examples, rather than updating the weights one data point at a time.

In [ ]:
class Perceptron:
    def __init__(self,lr=1e-3):
        # store the learning rate
        self.lr = lr

        # initialize the weights to small, normally-distributed values
        self.w = torch.normal(mean=0, std=0.01, size=(2,))

        # initialize the bias to zero
        self.b = torch.zeros(1)

    def train_step(self,X:torch.Tensor,y:torch.Tensor) -> None:
        """ Apply the first update rule shown in lecture.
            Arguments:
             X: data matrix of shape (N,2)
             y: labels of shape (N,) 
        """
        yhat = self.predict(X)
        error = yhat - y 
        grad_w = (X.T @ error) / X.shape[0]
        grad_b = torch.mean(error)
        self.w -= self.lr * grad_w
        self.b -= self.lr * grad_b

    def predict(self,X:torch.Tensor) -> torch.Tensor:
        """ Calculate model prediction for all data points.
            Arguments:
             X: data matrix of shape (N,2)   
            Returns:
             Predicted labels (0 or 1) of shape (N,)
        """
        Z = (X @ self.w + self.b)
        return (Z>0).float()
    
    def score(self,X:torch.Tensor,y:torch.Tensor) -> torch.Tensor:
        """ Calculate model accuracy
            Arguments:
             X: data matrix of shape (N,2)   
             y: labels of shape (N,)
            Returns:
             Accuracy score
        """
        return torch.mean((self.predict(X) == y).float())


Run the following code to train the model and print out the accuracy at each step.

In [141]:
lr = 0.0005
epochs = 100
model = Perceptron(lr)
for i in range(epochs):
    model.train_step(X,y)
    print(f'step {i}: {model.score(X,y)}')

step 0: 0.7710280418395996
step 1: 0.827102780342102
step 2: 0.8598130941390991
step 3: 0.8878504633903503
step 4: 0.9158878326416016
step 5: 0.9065420627593994
step 6: 0.9065420627593994
step 7: 0.9112149477005005
step 8: 0.9112149477005005
step 9: 0.9205607771873474
step 10: 0.9252336621284485
step 11: 0.9252336621284485
step 12: 0.9252336621284485
step 13: 0.9252336621284485
step 14: 0.9252336621284485
step 15: 0.9299065470695496
step 16: 0.9299065470695496
step 17: 0.9252336621284485
step 18: 0.9299065470695496
step 19: 0.9299065470695496
step 20: 0.9252336621284485
step 21: 0.9299065470695496
step 22: 0.9299065470695496
step 23: 0.9252336621284485
step 24: 0.9299065470695496
step 25: 0.9299065470695496
step 26: 0.9252336621284485
step 27: 0.9299065470695496
step 28: 0.9299065470695496
step 29: 0.9252336621284485
step 30: 0.9299065470695496
step 31: 0.9299065470695496
step 32: 0.9299065470695496
step 33: 0.9299065470695496
step 34: 0.9345794320106506
step 35: 0.9345794320106506
ste

Run the training multiple times.  Is the training the same each time, or does it vary?  Why?

It varies. It is finding different local minimums each time and is unsuccessful in finding a global minimum

Play with the learning rate and number of epochs to find the best setting.

LR ~= 0.001 - 0.0005 was good, epochs peaked around 100.